In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

from pathlib import Path
import torch

def get_clip_score(d):
    s = n = 0
    for p in Path(d).rglob('*.pt'):
        try:
            s += torch.load(p, map_location='cpu')['clip_score'].float().mean().item()
            n += 1
        except Exception:
            continue
    if n == 0:
        raise ValueError(f'No clip_score found under: {d}')
    return s / n, n  # (average, file_count)


In [5]:
pt_dirs = [
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_2/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_3/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_4/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_7/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_8/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_9/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_10/which_0',
           'samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0',
           
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    

100%|██████████| 30001/30001 [00:18<00:00, 1634.43it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0 23.985039418562792 0.31077868836522105


100%|██████████| 30001/30001 [00:18<00:00, 1613.69it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0 23.125421109208048 0.310546322307984


100%|██████████| 30001/30001 [00:18<00:00, 1654.68it/s]


...

samplings/SANA/4.5/3/Dual-Solver/30000/which_4/which_0 21.067437366083993 0.310162083007892


100%|██████████| 30001/30001 [00:17<00:00, 1691.04it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0 23.220470972418752 0.30901574394901593


100%|██████████| 7885/7885 [00:04<00:00, 1739.71it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0 26.397021486048743 0.3103408288034307


In [3]:
pt_dirs = [
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_0/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_1/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_2/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_3/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_4/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_5/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_6/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_7/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_8/which_0',
        #    'samplings/SANA/4.5/6/Dual-Solver/30000/which_9/which_0',
           'samplings/SANA/4.5/6/Dual-Solver/30000/which_10/which_0',
           'samplings/SANA/4.5/6/Dual-Solver/30000/which_11/which_0',
           'samplings/SANA/4.5/6/Dual-Solver/30000/which_12/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)

100%|██████████| 30001/30001 [00:19<00:00, 1501.27it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/which_10/which_0 23.492993677006893 0.31493527757724127


100%|██████████| 30001/30001 [00:21<00:00, 1405.75it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/which_11/which_0 21.42050369918468 0.3143629016896089


In [3]:
pt_dirs = ['samplings/SANA/2.5/3/Dual-Solver/30000/multi1_0',
           'samplings/SANA/2.5/4/Dual-Solver/30000/multi1_0',
           'samplings/SANA/2.5/5/Dual-Solver/30000/multi1_0',
           'samplings/SANA/2.5/6/Dual-Solver/30000/multi1_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    

100%|██████████| 30001/30001 [00:15<00:00, 1942.83it/s]


samplings/SANA/2.5/3/Dual-Solver/30000/multi1_0 22.3470025351823 0.3101592518766721


100%|██████████| 30001/30001 [00:13<00:00, 2244.38it/s]


samplings/SANA/2.5/4/Dual-Solver/30000/multi1_0 20.466995083363315 0.3135922824462255


100%|██████████| 30001/30001 [00:11<00:00, 2622.96it/s]


samplings/SANA/2.5/5/Dual-Solver/30000/multi1_0 21.044487644947196 0.3144221347351869


100%|██████████| 30001/30001 [00:10<00:00, 2740.82it/s]


samplings/SANA/2.5/6/Dual-Solver/30000/multi1_0 21.92791941607851 0.31462041897177695


In [6]:
pt_dirs = ['samplings/SANA/4.5/3/BNS-Solver/30000/bns_0',
           'samplings/SANA/4.5/4/BNS-Solver/30000/bns_0',
           'samplings/SANA/4.5/5/BNS-Solver/30000/bns_0',
           'samplings/SANA/4.5/6/BNS-Solver/30000/bns_0',
            ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    
# 100%|██████████| 30001/30001 [00:24<00:00, 1228.07it/s]
# samplings/SANA/4.5/4/BNS-Solver/30000/bns_0 26.375227349961392 0.30517675228913627    
# 100%|██████████| 30001/30001 [00:28<00:00, 1050.32it/s]
# samplings/SANA/4.5/5/BNS-Solver/30000/bns_0 21.04985502420982 0.3093961792945862
# 100%|██████████| 30001/30001 [00:19<00:00, 1554.89it/s]
# samplings/SANA/4.5/6/BNS-Solver/30000/bns_0 20.663592319863255 0.31136821121176084

100%|██████████| 30001/30001 [00:10<00:00, 2780.62it/s]


samplings/SANA/4.5/3/BNS-Solver/30000/bns_0 48.166845809745155 0.2947148498972257


100%|██████████| 30001/30001 [00:10<00:00, 2993.23it/s]


samplings/SANA/4.5/4/BNS-Solver/30000/bns_0 26.375227349961392 0.30517675228913627


100%|██████████| 30001/30001 [00:09<00:00, 3067.58it/s]


samplings/SANA/4.5/5/BNS-Solver/30000/bns_0 21.04985502420982 0.3093961792945862


100%|██████████| 30001/30001 [00:09<00:00, 3069.52it/s]


samplings/SANA/4.5/6/BNS-Solver/30000/bns_0 20.663592319863255 0.31136821121176084


In [5]:
pt_dirs = ['samplings/SANA/4.5/3/DS-Solver_Flow/30000/ds_0',
           'samplings/SANA/4.5/4/DS-Solver_Flow/30000/ds_0',
           'samplings/SANA/4.5/5/DS-Solver_Flow/30000/ds_0',
           'samplings/SANA/4.5/6/DS-Solver_Flow/30000/ds_0',
            ]
for pt_dir in pt_dirs:
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    


100%|██████████| 30001/30001 [00:10<00:00, 2845.48it/s]


samplings/SANA/4.5/3/DS-Solver_Flow/30000/ds_0 48.6522788950582 0.2945741979678472


100%|██████████| 30001/30001 [00:10<00:00, 2876.63it/s]


samplings/SANA/4.5/4/DS-Solver_Flow/30000/ds_0 29.150639246893718 0.3078971952001254


100%|██████████| 30001/30001 [00:10<00:00, 2975.36it/s]


samplings/SANA/4.5/5/DS-Solver_Flow/30000/ds_0 21.663524320343072 0.31273227626681327


100%|██████████| 30001/30001 [00:11<00:00, 2550.75it/s]


samplings/SANA/4.5/6/DS-Solver_Flow/30000/ds_0 20.6583094264862 0.31406922237674395


In [3]:
pt_dirs = ['samplings/SANA/4.5/3/Dual-Solver/30000/multi1_0',
           'samplings/SANA/4.5/4/Dual-Solver/30000/multi1_0',
           'samplings/SANA/4.5/5/Dual-Solver/30000/multi1_0',
           'samplings/SANA/4.5/6/Dual-Solver/30000/multi1_0',
          ]
for pt_dir in pt_dirs:
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    


100%|██████████| 30001/30001 [00:12<00:00, 2463.07it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/multi1_0 22.66329372767086 0.3114113017320633


100%|██████████| 30001/30001 [00:10<00:00, 2880.77it/s]


samplings/SANA/4.5/4/Dual-Solver/30000/multi1_0 21.717621822936678 0.31374961573680243


100%|██████████| 30001/30001 [00:10<00:00, 2856.66it/s]


samplings/SANA/4.5/5/Dual-Solver/30000/multi1_0 21.77566302707112 0.3150470300455888


100%|██████████| 30001/30001 [00:11<00:00, 2661.18it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/multi1_0 22.590633272346224 0.31593455819884936


In [4]:
pt_dirs = ['samplings/SANA/4.5/3/Dual-Solver/30000/traj_0',
           'samplings/SANA/4.5/4/Dual-Solver/30000/traj_0',
           'samplings/SANA/4.5/5/Dual-Solver/30000/traj_0',
           'samplings/SANA/4.5/6/Dual-Solver/30000/traj_0',
          ]
for pt_dir in pt_dirs:
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    


100%|██████████| 30001/30001 [00:10<00:00, 2795.39it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/traj_0 48.97376895382894 0.2935381324748198


100%|██████████| 30001/30001 [00:09<00:00, 3038.51it/s]


samplings/SANA/4.5/4/Dual-Solver/30000/traj_0 22.927048832026514 0.30781269566218056


100%|██████████| 30001/30001 [00:10<00:00, 2780.60it/s]


samplings/SANA/4.5/5/Dual-Solver/30000/traj_0 21.41343844194313 0.30878774306774137


100%|██████████| 30001/30001 [00:10<00:00, 2740.65it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/traj_0 20.579232937739334 0.3114000505844752
